# Преобразование текстовых данных

### Импорт библиотек, установка констант

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
AG_NEWS_TRAIN = 'https://www.dropbox.com/scl/fi/ok9tt9prs7yl9o974ipgn/ag_news_train.csv?rlkey=wlh19mk7q5cn3v6n7zluzx2bn&st=y5i93t3v&dl=1'
AG_NEWS_TEST = 'https://www.dropbox.com/scl/fi/5abmk0nnzz4tqyajqhfwu/ag_news_test.csv?rlkey=yu4ngbm1yccvvcgs1u0htpav4&st=3cv3suy2&dl=1'

### Загрузка данных

In [ ]:
data_train = pd.read_csv(AG_NEWS_TRAIN, sep='\t')
data_test = pd.read_csv(AG_NEWS_TEST, sep='\t')

In [ ]:
labels_train = data_train['label']
labels_test = data_test['label']

In [ ]:
len(labels_train)

120000

### Обучение модели

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
def train_eval_model(train_X, test_X, train_y, test_y):
    model = LogisticRegression(max_iter=500)
    model.fit(train_X, train_y)

    train_pred = model.predict(train_X)
    test_pred = model.predict(test_X)

    train_acc = accuracy_score(train_y, train_pred)
    test_acc = accuracy_score(test_y, test_pred)

    print('Train accuracy:', round(train_acc, 3))
    print('Test accuracy: ', round(test_acc, 3))

### Токенизация

Токенизация - это деление текста на токены (в простейшем случае отдельные слова).

Токенизацию проще всего реализовать с помощью [регулярных выражений](https://docs.python.org/3/library/re.html). Но можно и без них.

In [ ]:
import re
from string import punctuation

In [ ]:
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
# this two functions do the same job

def tokenize(text):
    for p in punctuation:
        text = text.replace(p, ' ')

    text = text.strip().split()
    return text

def tokenize(text):
    reg = re.compile(r'\w+')
    return reg.findall(text)

In [ ]:
data_train['text'][50]

'Making Your Insurer Pay If Hurricane Charley blows your house down, how can you make your insurance company pay?'

In [ ]:
print(tokenize(data_train['text'][50]))

['Making', 'Your', 'Insurer', 'Pay', 'If', 'Hurricane', 'Charley', 'blows', 'your', 'house', 'down', 'how', 'can', 'you', 'make', 'your', 'insurance', 'company', 'pay']


__`nltk`__ - это огромная библиотека для работы с текстом (сравнимо с `numpy` для матриц). В ней реализованы методы для [токенизации](https://www.nltk.org/api/nltk.tokenize.html), [лемматизации](https://www.nltk.org/api/nltk.stem.wordnet.html), [стемминга](https://www.nltk.org/api/nltk.stem.html) и многого другого.

In [ ]:
from nltk.tokenize import wordpunct_tokenize

print(wordpunct_tokenize(data_train['text'][50]))

['Making', 'Your', 'Insurer', 'Pay', 'If', 'Hurricane', 'Charley', 'blows', 'your', 'house', 'down', ',', 'how', 'can', 'you', 'make', 'your', 'insurance', 'company', 'pay', '?']


Приведем все к нижнему регистру и токенизируем.

In [ ]:
data_tok_train = [tokenize(t.lower()) for t in data_train['text']]
data_tok_test = [tokenize(t.lower()) for t in data_test['text']]

In [ ]:
print(data_tok_train[:2])

[['wall', 'st', 'bears', 'claw', 'back', 'into', 'the', 'black', 'reuters', 'reuters', 'short', 'sellers', 'wall', 'street', 's', 'dwindling', 'band', 'of', 'ultra', 'cynics', 'are', 'seeing', 'green', 'again'], ['carlyle', 'looks', 'toward', 'commercial', 'aerospace', 'reuters', 'reuters', 'private', 'investment', 'firm', 'carlyle', 'group', 'which', 'has', 'a', 'reputation', 'for', 'making', 'well', 'timed', 'and', 'occasionally', 'controversial', 'plays', 'in', 'the', 'defense', 'industry', 'has', 'quietly', 'placed', 'its', 'bets', 'on', 'another', 'part', 'of', 'the', 'market']]


### Удаление стоп-слов

In [ ]:
from nltk.corpus import stopwords

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
stop_words = stopwords.words('english')
stop_words += ['ap', 'us', 'u', '39']
print(stop_words)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
def remove_stopwords(tokenized_texts):
    clear_texts = []

    for words in tokenized_texts:
        clear_texts.append([word for word in words if word not in stop_words])

    return clear_texts

In [ ]:
data_tok_train = remove_stopwords(data_tok_train)
data_tok_test = remove_stopwords(data_tok_test)

In [ ]:
print(data_train['text'][50])
print(data_tok_train[50])

Making Your Insurer Pay If Hurricane Charley blows your house down, how can you make your insurance company pay?
['making', 'insurer', 'pay', 'hurricane', 'charley', 'blows', 'house', 'make', 'insurance', 'company', 'pay']


### Стемминг

**Стемминг** –  это процесс нахождения основы слова. В результате его применения однокоренные слова преобразуются к одинаковому виду.

**Примеры стемминга:**

| Word        | Stem           |
| ----------- |:-------------:|
| вагон | вагон |
| вагона | вагон |
| вагоне | вагон |
| вагонов | вагон |
| вагоном | вагон |
| вагоны | вагон |
| важная | важн |
| важнее | важн |
| важнейшие | важн |
| важнейшими | важн |
| важничал | важнича |
| важно | важн |

Для английского языка – `nltk.stem.PorterStemmer` или `nltk.stem.SnowballStemmer`.   
Для русского – `nltk.stem.SnowballStemmer(language="russian")`.

In [ ]:
from nltk.stem import PorterStemmer

def stem_text(tokenized_texts):
    stemmed_data = []
    stemmer = PorterStemmer()

    for words in tqdm(tokenized_texts):
        stemmed_words = [stemmer.stem(word) for word in words]
        stemmed_data.append(stemmed_words)

    return stemmed_data

In [ ]:
stemmed_train = stem_text(data_tok_train)
stemmed_test = stem_text(data_tok_test)

  0%|          | 0/120000 [00:00<?, ?it/s]

  0%|          | 0/7600 [00:00<?, ?it/s]

In [ ]:
print(data_tok_train[50])

['making', 'insurer', 'pay', 'hurricane', 'charley', 'blows', 'house', 'make', 'insurance', 'company', 'pay']


In [ ]:
print(stemmed_train[50])

['make', 'insur', 'pay', 'hurrican', 'charley', 'blow', 'hous', 'make', 'insur', 'compani', 'pay']


In [ ]:
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x, token_pattern=None, lowercase=False, min_df=4)
tfidf_vectorizer.fit(stemmed_train);

In [ ]:
tfidf_vectorizer.transform(stemmed_train[:1]).shape

(1, 20095)

In [ ]:
tfidf_stemmed_train = tfidf_vectorizer.transform(stemmed_train)
tfidf_stemmed_test = tfidf_vectorizer.transform(stemmed_test)

In [ ]:
train_eval_model(tfidf_stemmed_train, tfidf_stemmed_test, labels_train, labels_test)

Train accuracy: 0.932
Test accuracy:  0.911


### Лемматизация

__Лемматизация__ — процесс приведения слова к его нормальной форме (**лемме**):
- для существительных — именительный падеж, единственное число;
- для прилагательных — именительный падеж, единственное число, мужской род;
- для глаголов, причастий, деепричастий — глагол в инфинитиве.

Для английского языка – `nltk.stem.WordNetLemmatizer`.   
Для русского – `pymorphy2.MorphAnalyzer`.

In [ ]:
from nltk.stem import WordNetLemmatizer

def lemmatize_text(tokenized_texts):
    lemmatized_data = []
    lemmatizer = WordNetLemmatizer()

    for _, words in enumerate(tqdm(tokenized_texts)):
        lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
        lemmatized_data.append(lemmatized_words)

    return lemmatized_data

In [ ]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
lemmatized_train = lemmatize_text(data_tok_train)
lemmatized_test = lemmatize_text(data_tok_test)

  0%|          | 0/120000 [00:00<?, ?it/s]

  0%|          | 0/7600 [00:00<?, ?it/s]

In [ ]:
print(data_tok_train[50])

['making', 'insurer', 'pay', 'hurricane', 'charley', 'blows', 'house', 'make', 'insurance', 'company', 'pay']


In [ ]:
print(lemmatized_train[50])

['making', 'insurer', 'pay', 'hurricane', 'charley', 'blow', 'house', 'make', 'insurance', 'company', 'pay']


In [ ]:
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x, token_pattern=None, lowercase=False, min_df=4)
tfidf_vectorizer.fit(lemmatized_train);

In [ ]:
tfidf_vectorizer.transform(lemmatized_train[:1]).shape

(1, 25844)

In [ ]:
tfidf_lemmatized_train = tfidf_vectorizer.transform(lemmatized_train)
tfidf_lemmatized_test = tfidf_vectorizer.transform(lemmatized_test)

In [ ]:
train_eval_model(tfidf_lemmatized_train, tfidf_lemmatized_test, labels_train, labels_test)

Train accuracy: 0.937
Test accuracy:  0.913


## Тематическое моделирование

**Тематическая модель** (topic model) — модель коллекции текстовых документов, которая определяет, к каким темам относится каждый документ коллекции. Алгоритм построения тематической модели получает на входе коллекцию текстовых документов. На выходе для каждого документа выдаётся числовой вектор, составленный из оценок степени принадлежности данного документа каждой из тем. Размерность этого вектора, равная числу тем, может либо задаваться на входе, либо определяться моделью автоматически.

**Тематическое моделирование** (topic modeling) — построение тематической модели.

Методы тематического моделирования можно разделить на две основных группы – алгебраические и вероятностные.  
**Латентно-семантический анализ (LSA)** относится к алгебраическим
методам, а среди вероятностных наиболее популярным является **латентное размещение Дирихле (LDA)**. Так как LSA и LDA основаны на очень разных математических процедурах,
то, очевидно, что в зависимости от типа входных текстовых данных они будут иметь разную степень успеха. При этом алгоритм использования их в рамках прикладной задачи может быть достаточно схож.

<img src="https://upload.wikimedia.org/wikipedia/commons/d/d5/Тематическая_модель.png" width="700" height="700">

In [ ]:
%%capture
!pip install -U gensim

In [ ]:
from gensim.models.ldamodel import LdaModel
from gensim.corpora.dictionary import Dictionary

id2word = Dictionary(lemmatized_train)

In [ ]:
corpus = [id2word.doc2bow(text) for text in lemmatized_train]

In [ ]:
# Обучаем модель LDA
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=8
)

In [ ]:
# визуализируем наиболее популярные слова в каждой теме
topics = lda_model.show_topics(formatted=False)

# преобразуем выход метода
topics = {
    f'topic {i}': [pair[0] for pair in topic[1]] for i, topic in enumerate(topics)
    }

pd.DataFrame(topics)

,topic 0,topic 1,topic 2,topic 3,topic 4,topic 5,topic 6,topic 7
0,season,gt,new,quot,said,said,oil,new
1,team,lt,microsoft,president,year,iraq,reuters,game
2,win,b,company,court,billion,reuters,price,year
3,game,reuters,software,bush,million,minister,stock,red
4,coach,fullquote,service,say,company,official,new,one
5,first,n,internet,said,deal,india,dollar,first
6,point,com,search,drug,inc,killed,market,player
7,night,target,computer,state,palestinian,iraqi,york,sox
8,sport,stock,technology,oracle,1,two,share,boston
9,last,ticker,system,election,plan,police,rate,2
